# 04 - D-FINE-N (depot officiel Peterande/D-FINE)

Config `dfine_hgnetv2_n_coco`, poids COCO `dfine_n_coco.pth` en fine-tuning. Memes transforms ajoutees que pour RT-DETR.

In [ ]:
# --- Installation (le depot installe ses propres dependances au 1er appel) ---
!pip install -q pycocotools wandb

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
import os, sys
REPO_DIR = "/content/aphids_detection"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/EmmaDub/aphids_detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__)

In [ ]:
# --- Configuration ---
# Les chemins par defaut sont ceux de aphids_det/config.py. Pour les changer,
# decommenter et adapter, puis relancer cfg.refresh().
from pathlib import Path
import aphids_det.config as cfg

# cfg.BASE_DIR  = Path("/content/drive/MyDrive/.../tuile_viz02_640_128")
# cfg.OUT_DIR   = Path("/content/drive/MyDrive/.../puceron_model_article/data")
# cfg.EPOCHS    = 30
# cfg.USE_WANDB = True

cfg.refresh()
cfg.summary()

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Suivi W&B (facultatif : mettre cfg.USE_WANDB = False pour s'en passer) ---
if cfg.USE_WANDB:
    import wandb
    wandb.login()
    os.environ["WANDB_PROJECT"] = cfg.WANDB_PROJECT
    print("W&B -> projet", cfg.WANDB_PROJECT)

In [ ]:
from aphids_det.runners import dfine_runner
df = dfine_runner.run_cv()

In [ ]:
# --- Etat du CSV de benchmark ---
import pandas as pd
d = pd.read_csv(cfg.CSV_CV)
print(d.groupby("modele")["fold"].count().to_string(), "\n")
d[["modele", "fold", "map50_macro", "map5095_macro", "latency_cpu_ms",
   "n_params_M", "train_time_s"]].tail(10)